# NutriLens — Plate Classifier (Food-101)

Your first training run. Produces a ResNet-50 fine-tuned on Food-101 that recognises 101 food categories. This is the warm-start for plate recognition; detector (Mask R-CNN) training plugs in later once we have mask-labeled data.

**Before you hit Run All:**
1. `Runtime` → `Change runtime type` → **Hardware accelerator: T4 GPU**.
2. Have your R2 credentials ready (see `docs/INFRASTRUCTURE.md`).
3. Expect ~45–90 minutes wallclock on a free T4.

If anything in the first cell fails, your runtime doesn't have a GPU. Fix that before continuing — CPU training on this dataset takes days.

## 0. Sanity check — GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 1. Clone the repo and install

Edit the URL below if your GitHub username/repo differs. Public repos work without a token.

In [ ]:
REPO_URL = 'https://github.com/jaelricco/nutrilens-ml.git'  # <-- edit me

# %cd / first so re-running this cell can't delete our own working directory.
%cd /
!rm -rf /content/nutrilens-ml
!git clone --depth 1 {REPO_URL} /content/nutrilens-ml
%cd /content/nutrilens-ml

from pathlib import Path
assert Path('/content/nutrilens-ml/pyproject.toml').is_file(), (
    'Clone failed — no pyproject.toml in /content/nutrilens-ml. '
    'Is the repo public? Is REPO_URL correct?'
)
print('clone OK')

In [ ]:
# Non-editable install + verbose output so any failure is visible.
!pip install --upgrade pip
!pip install /content/nutrilens-ml boto3 wandb onnxscript

import importlib, sys
# Drop any cached failed import from a previous run.
for mod in list(sys.modules):
    if mod == 'nutrilens_ml' or mod.startswith('nutrilens_ml.'):
        del sys.modules[mod]
import nutrilens_ml  # noqa: F401 — fails loudly if install didn't take
print('nutrilens_ml import OK')

## 2. R2 credentials

You'll be prompted. Values come from your Cloudflare R2 API token (see `docs/INFRASTRUCTURE.md`).

In [ ]:
import os, getpass

os.environ['S3_BUCKET'] = input('R2 bucket name [nutrilens-ml]: ').strip() or 'nutrilens-ml'
os.environ['S3_ENDPOINT'] = input('R2 endpoint URL (https://<account-id>.r2.cloudflarestorage.com): ').strip()
os.environ['S3_REGION'] = 'auto'
os.environ['S3_ACCESS_KEY_ID'] = getpass.getpass('R2 Access Key ID: ')
os.environ['S3_SECRET_ACCESS_KEY'] = getpass.getpass('R2 Secret Access Key: ')

wandb_key = getpass.getpass('W&B API key (optional — press Enter to skip): ')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
print('credentials set')

## 3. Download Food-101

About 5 GB. First run takes ~3 minutes on Colab's network.

In [ ]:
from pathlib import Path
from nutrilens_ml.data.food101 import build_food101_datasets

train_ds, val_ds, class_names = build_food101_datasets(root=Path('/content/data'), image_size=224)
print(f'{len(train_ds)} train / {len(val_ds)} val / {len(class_names)} classes')

## 4. Train

Default: 5 epochs @ batch 64. At the end you'll see `val/top1` around 0.55–0.65 — reasonable for a first run. Bump `epochs` later to push higher; the Phase 3 release bar is `top5 >= 0.60`.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

from nutrilens_ml.training.plate_classifier import TrainConfig, train_classifier

cfg = TrainConfig(
    num_classes=len(class_names),
    epochs=5,
    batch_size=64,
    lr=1e-4,
    run_dir=Path('/content/nutrilens-ml/runs/plate-classifier-v0'),
    class_names=class_names,
    wandb=bool(os.environ.get('WANDB_API_KEY')),
)
best_ckpt = train_classifier(train_ds, val_ds, cfg)
print('best checkpoint:', best_ckpt)

## 4.5. Backup checkpoint to R2 (seatbelt)

Training is the expensive step. Upload `best.pt` to R2 *before* export so a runtime death from here on is recoverable — you can pull the checkpoint back from R2 in a fresh session instead of re-training.

In [ ]:
import boto3, os
from pathlib import Path

_s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT'],
    aws_access_key_id=os.environ['S3_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['S3_SECRET_ACCESS_KEY'],
    region_name='auto',
)

# Stable, overwritable recovery prefix. Final versioned upload happens later in cell 6.
_backup_prefix = 'models/plate-classifier/_wip'
_class_names_path = cfg.run_dir / 'class_names.json'

for _local in (best_ckpt, _class_names_path):
    _key = f'{_backup_prefix}/{Path(_local).name}'
    _s3.upload_file(str(_local), os.environ['S3_BUCKET'], _key)
    print('backed up ->', _key)

print('\nIf Colab dies from here on, recover with: s3://<bucket>/' + _backup_prefix)

## 4.6. Recover from a lost runtime (skip on a normal run)

Use this **only** if Colab killed your runtime *after* training (cell 4.5 successfully uploaded `best.pt` to R2) but *before* the export step finished.

What it does: pulls the `best.pt` and `class_names.json` you backed up to `models/plate-classifier/_wip/` back into this fresh runtime, and reconstructs the `cfg`, `class_names`, `best_ckpt` variables so the export and final-upload cells just work.

**Skip this cell on a normal run** — it's a no-op safety net for resuming after a crash.

Order of operations to recover:
1. Run sections 0, 1, 2 (GPU check, clone, install, credentials).
2. **Skip** sections 3 (Food-101 download) and 4 (training) — you don't need them.
3. Run this cell.
4. Continue from section 5 (export) onward.

In [ ]:
import os, json, boto3
from pathlib import Path
from nutrilens_ml.training.plate_classifier import TrainConfig

recovery_dir = Path('/content/nutrilens-ml/runs/plate-classifier-v0')
recovery_dir.mkdir(parents=True, exist_ok=True)

_s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT'],
    aws_access_key_id=os.environ['S3_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['S3_SECRET_ACCESS_KEY'],
    region_name='auto',
)

_backup_prefix = 'models/plate-classifier/_wip'
for _name in ('best.pt', 'class_names.json'):
    _dest = recovery_dir / _name
    _s3.download_file(os.environ['S3_BUCKET'], f'{_backup_prefix}/{_name}', str(_dest))
    print('recovered ->', _dest)

best_ckpt = recovery_dir / 'best.pt'
class_names = json.loads((recovery_dir / 'class_names.json').read_text())
cfg = TrainConfig(
    num_classes=len(class_names),
    run_dir=recovery_dir,
    class_names=class_names,
)
print(f'\nready: {len(class_names)} classes, ckpt at {best_ckpt}')
print('continue from section 5 (Export to ONNX)')

## 5. Export to ONNX

Produces the portable artifact. On your Mac later, `coremltools` turns this into a `.mlpackage` for Xcode.

In [ ]:
import torch
from nutrilens_ml.models.plate_classifier import build_plate_classifier
from nutrilens_ml.export.convert import export_to_onnx, parity_check

# weights_only=False: we saved this file ourselves a few minutes ago.
state = torch.load(best_ckpt, map_location='cpu', weights_only=False)
model = build_plate_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(state['model'])
model.eval()

example = torch.randn(1, 3, 224, 224)
onnx_path = cfg.run_dir / 'plate-classifier.onnx'
export_to_onnx(model, example, onnx_path, input_names=('image',), output_names=('logits',))

delta = parity_check(model, onnx_path, example, tol=1e-3)
print(f'ONNX parity: max|Δ| = {delta:.2e}')

## 6. Upload to R2

In [ ]:
import subprocess, json
from nutrilens_ml.eval.registry import ModelManifest, upload_model

git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()

manifest = ModelManifest(
    task='plate-classifier',
    version='0.1.0',
    metrics={'top1': state['top1'], 'top5': state['top5']},
    dataset_hash='food101-public',
    training_git_sha=git_sha,
    notes='First training run from Colab notebook.',
)

class_names_path = cfg.run_dir / 'class_names.json'
key_prefix = upload_model(
    bucket=os.environ['S3_BUCKET'],
    prefix='models',
    manifest=manifest,
    artifacts=[best_ckpt, onnx_path, class_names_path],
    endpoint=os.environ['S3_ENDPOINT'],
    region='auto',
)
print('uploaded to:', key_prefix)
print('\nmetrics:', json.dumps(manifest.metrics, indent=2))

## 7. Download the ONNX to your Mac (optional)

Colab's `/content/` is wiped when the runtime ends, but the artifact is safely in R2. If you want a local copy now:

In [ ]:
from google.colab import files
files.download(str(onnx_path))
files.download(str(class_names_path))

## Done

Next steps:
- On your Mac: `pip install -e '.[export]'` then `nutrilens-ml export plate --checkpoint <downloaded .onnx>` to get a `.mlpackage`.
- Drag the `.mlpackage` into Xcode.
- Commit a scorecard under `docs/scorecards/plate-classifier-0.1.0.md` with the metrics printed above.